In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import numpy as np
from torch.utils.data import DataLoader, Subset

In [4]:
from torch.utils.data import Dataset
from PIL import Image
from pathlib import Path

class CelebALandmarksDataset(Dataset):
    def __init__(self, landmarks_path, img_dir, transform=None):
        self.img_dir = Path(img_dir)
        self.transform = transform
        self.samples = []
        with open(landmarks_path) as f:
            lines = f.readlines()
        # Skip line 0 (count), line 1 (header)
        for line in lines[2:]:
            parts = line.split()
            if len(parts) >= 11:
                fname = parts[0]
                coords = [float(x) for x in parts[1:11]]
                self.samples.append((fname, coords))

        # Keep only samples where the image file exists
        self.samples = [(f, c) for f, c in self.samples if (self.img_dir / f).exists()]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        fname, coords = self.samples[idx]
        img = Image.open(self.img_dir / fname).convert("RGB")
        landmarks = torch.tensor(coords, dtype=torch.float32)
        if self.transform:
            img = self.transform(img)
        return img, landmarks

# Paths
landmarks_path = "/home/toru2/Amara/Deep_learning/dl_lab345.ipynb/dataset/landmarks/list_landmarks_align_celeba.txt"
img_dir = "/home/toru2/Amara/Deep_learning/dl_lab345.ipynb/dataset/img_align_celeba"

transform = torchvision.transforms.Compose([
    torchvision.transforms.PILToTensor(),
])
full_dataset = CelebALandmarksDataset(landmarks_path, img_dir, transform=transform)

train_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_size
train_set, test_set = torch.utils.data.random_split(
    full_dataset, [train_size, test_size],
    generator=torch.Generator().manual_seed(42)
)
print(len(train_set), len(test_set))

58220 14555


In [5]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_chanels, **kwargs):
        super(ConvBlock, self).__init__()

        self.conv = nn.Conv2d(
            in_channels,
            out_chanels,
            bias=False,
            **kwargs
        )

        self.bn = nn.BatchNorm2d(out_chanels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(self.bn(self.conv(x)))


class InceptionBlock(nn.Module):
    def __init__(self, in_channels, out_1x1, red_3x3, out_3x3,
                 red_5x5, out_5x5, out_pool):

        super(InceptionBlock, self).__init__()

        # 1x1 branch
        self.branch1 = ConvBlock(
            in_channels,
            out_1x1,
            kernel_size=1
        )

        # 1x1 -> 3x3 branch
        self.branch2 = nn.Sequential(
            ConvBlock(in_channels, red_3x3, kernel_size=1),
            ConvBlock(red_3x3, out_3x3, kernel_size=3, padding=1)
        )

        # 1x1 -> 3x3 -> 3x3 (replacement for 5x5)
        self.branch3 = nn.Sequential(
            ConvBlock(in_channels, red_5x5, kernel_size=1),
            ConvBlock(red_5x5, out_5x5, kernel_size=3, padding=1),
            ConvBlock(out_5x5, out_5x5, kernel_size=3, padding=1)
        )

        # pooling branch
        self.branch4 = nn.Sequential(
            nn.MaxPool2d(kernel_size=3, stride=1, padding=1),
            ConvBlock(in_channels, out_pool, kernel_size=1)
        )

        self.out_channels = out_1x1 + out_3x3 + out_5x5 + out_pool

    def forward(self, x):

        y1 = self.branch1(x)
        y2 = self.branch2(x)
        y3 = self.branch3(x)
        y4 = self.branch4(x)

        return torch.cat([y1, y2, y3, y4], dim=1)


class InceptionResBlock(nn.Module):
    def __init__(self, in_channels):
        super(InceptionResBlock, self).__init__()

        self.block = InceptionBlock(
            in_channels,
            64, 96, 128,
            16, 32,
            32
        )

        self.proj = nn.Conv2d(
            self.block.out_channels,
            in_channels,
            kernel_size=1,
            bias=False
        )

        self.bn = nn.BatchNorm2d(in_channels)

    def forward(self, x):

        out = self.block(x)
        out = self.proj(out)
        out = self.bn(out)

        return F.relu(out + x)

In [6]:
class InceptionLandmarkModel(nn.Module):
    def __init__(self, num_landmarks=5):
        super(InceptionLandmarkModel, self).__init__()

        # Initial convolution blocks
        self.conv_blocks = nn.Sequential(
            ConvBlock(3, 64, kernel_size=7, stride=2, padding=3),
            nn.MaxPool2d(3, stride=2, padding=1),

            ConvBlock(64, 128, kernel_size=3, padding=1),
            ConvBlock(128, 192, kernel_size=3, padding=1),
            nn.MaxPool2d(3, stride=2, padding=1)
        )

        # First group of inception blocks
        self.inception_block_group1 = nn.Sequential(
            InceptionBlock(192, 64, 96, 128, 16, 32, 32),
            InceptionBlock(256, 128, 128, 192, 32, 96, 64),
            nn.MaxPool2d(3, stride=2, padding=1)
        )

        # Residual inception blocks
        self.inception_res_blocks = nn.Sequential(
            InceptionResBlock(480),
            InceptionResBlock(480),
            InceptionResBlock(480)
        )

        # Final inception blocks
        self.inception_block_group2 = nn.Sequential(
            InceptionBlock(480, 256, 160, 320, 32, 128, 128),
            InceptionBlock(832, 256, 192, 320, 48, 128, 128)
        )

        # Output layers
        self.global_avg_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(0.4)
        self.fc = nn.Linear(832, num_landmarks * 2)

    def forward(self, x):

        x = self.conv_blocks(x)

        x = self.inception_block_group1(x)

        x = self.inception_res_blocks(x)

        x = self.inception_block_group2(x)

        x = self.global_avg_pool(x)

        x = torch.flatten(x, 1)

        x = self.dropout(x)

        x = self.fc(x)

        # Constrain normalized coordinate outputs to [0, 1]
        return torch.sigmoid(x)

In [7]:
batch_size = 8

subset_indices = list(range(batch_size))
sanity_dataset = Subset(train_set, subset_indices)

sanity_loader = DataLoader(
    sanity_dataset,
    batch_size=batch_size,
    shuffle=False
)

device = "cuda" if torch.cuda.is_available() else "cpu"

model = InceptionLandmarkModel(num_landmarks=5).to(device)

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

In [8]:
epochs = 500

model.train()

for epoch in range(epochs):

    for imgs, landmarks in sanity_loader:

        imgs = imgs.float().to(device) / 255.0
        landmarks = landmarks.to(device)

        # Normalize CelebA coordinates
        landmarks[:, 0::2] /= 178   # x coordinates
        landmarks[:, 1::2] /= 218   # y coordinates

        preds = model(imgs)

        loss = criterion(preds, landmarks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    if epoch % 50 == 0:
        print(f"Epoch {epoch}  Loss: {loss.item():.6f}")

Epoch 0  Loss: 0.023967
Epoch 50  Loss: 0.000391
Epoch 100  Loss: 0.000242
Epoch 150  Loss: 0.000177
Epoch 200  Loss: 0.000185
Epoch 250  Loss: 0.000108
Epoch 300  Loss: 0.000106
Epoch 350  Loss: 0.000129
Epoch 400  Loss: 0.000081
Epoch 450  Loss: 0.000059


In [9]:
print("Prediction:")
print(preds[0])

print("Target:")
print(landmarks[0])

Prediction:
tensor([0.4073, 0.5005, 0.6017, 0.5067, 0.4935, 0.6169, 0.4057, 0.7103, 0.5804,
        0.6941], device='cuda:0', grad_fn=<SelectBackward0>)
Target:
tensor([0.3989, 0.5046, 0.5955, 0.5046, 0.5000, 0.6147, 0.4045, 0.7064, 0.5899,
        0.7064], device='cuda:0')


In [10]:
for imgs, landmarks in sanity_loader:
    print(imgs.shape)       # expected [B,3,H,W]
    print(landmarks.shape)  # expected [B,10]
    
    break

torch.Size([8, 3, 218, 178])
torch.Size([8, 10])


Full Dataset training

In [12]:
from torch.utils.data import DataLoader

batch_size = 128

train_loader = DataLoader(
    train_set,
    batch_size=batch_size,
    shuffle=True,
    num_workers=8,
    pin_memory=True
)

test_loader = DataLoader(
    test_set,
    batch_size=batch_size,
    shuffle=False,
    num_workers=8,
    pin_memory=True
)

In [ ]:
device = "cuda"

model = InceptionLandmarkModel(num_landmarks=5).to(device)
criterion = torch.nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-5
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=3
)

epochs = 30

for epoch in range(epochs):

    model.train()
    train_loss = 0

    for imgs, landmarks in train_loader:

        imgs = imgs.to(device)
        landmarks = landmarks.to(device)

        # Normalize coordinates
        landmarks[:, 0::2] /= 178
        landmarks[:, 1::2] /= 218

        preds = model(imgs)

        loss = criterion(preds, landmarks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)



    model.eval()
    val_loss = 0

    with torch.no_grad():

        for imgs, landmarks in test_loader:

            imgs = imgs.to(device)
            landmarks = landmarks.to(device)

            landmarks[:, 0::2] /= 178
            landmarks[:, 1::2] /= 218

            preds = model(imgs)

            loss = criterion(preds, landmarks)

            val_loss += loss.item()

    val_loss /= len(test_loader)

    scheduler.step(val_loss)

    print(
        f"Epoch {epoch+1}/{epochs} | "
        f"Train Loss {train_loss:.5f} | "
        f"Val Loss {val_loss:.5f}"
    )

Saving model

In [ ]:
best_loss = float("inf")

if val_loss < best_loss:
    best_loss = val_loss
    torch.save(model.state_dict(), "best_landmark_model.pth")

In [ ]:
# Test fixed model on first 5 dataset images
from pathlib import Path
import torch
import torchvision.transforms as T
from PIL import Image
import matplotlib.pyplot as plt

# Paths
ckpt_path = Path('/home/toru2/Amara/Deep_learning/checkpoints/face_landmarks_fixed.pt')
img_dir = Path('/home/toru2/Amara/Deep_learning/dl_lab345.ipynb/dataset/img_align_celeba')

# Device + model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = InceptionLandmarkModel(num_landmarks=5).to(device)
model.load_state_dict(torch.load(ckpt_path, map_location=device))
model.eval()


# Preprocess (same as training)
IMG_W, IMG_H = 178, 218
transform = T.Compose([
    T.Resize((IMG_H, IMG_W)),
    T.ToTensor(),
])
scale = torch.tensor([IMG_W, IMG_H] * 5, device=device)

landmark_names = ['left_eye', 'right_eye', 'nose', 'left_mouth', 'right_mouth']

# First 5 images from folder
image_paths = sorted([p for p in img_dir.iterdir() if p.suffix.lower() in ['.jpg', '.jpeg', '.png']])[:10]
print(f'Using {len(image_paths)} images')

fig, axes = plt.subplots(1, len(image_paths), figsize=(4 * len(image_paths), 5))
if len(image_paths) == 1:
    axes = [axes]

for ax, path in zip(axes, image_paths):
    img = Image.open(path).convert('RGB')
    x = transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        pred = (model(x) * scale).squeeze(0).cpu().view(5, 2)

    img_np = transform(img).permute(1, 2, 0).numpy()
    ax.imshow(img_np)

    for i, (px, py) in enumerate(pred.tolist()):
        ax.scatter(px, py, c='red', s=25)
        ax.text(px + 2, py + 2, landmark_names[i], color='yellow', fontsize=7)

    ax.set_title(path.name)
    ax.set_xlim(0, IMG_W)
    ax.set_ylim(IMG_H, 0)
    ax.axis('off')

plt.tight_layout()
plt.show()